# Track B — Fase 1: Ekstraksi Semua Backbone (REVISI 15 Juli)

**BDC Satria Data 2026** | 5 foundation model beku → cache embedding di Drive

| | |
|---|---|
| Backbone | SigLIP2-base, SigLIP2-so400m, SigLIP1-base, DINOv3-ViT-L, DINOv3-ConvNeXt-B |
| Varian | non-TTA + TTA (hflip+vflip, dirata-rata di level embedding) |
| Split | train (26.527) + test (1.458) |
| Output | `emb_*.npy` + manifest `.json` di Drive |

> **Perbaikan di revisi ini:**
> - **`hf_xet` DIPASANG** (bukan di-uninstall). Hub 1.x butuh ini; tanpa `hf_xet`
>   download gagal `SignatureError`. *Uninstall-nya yang bikin gagal berulang 14–15 Juli.*
> - **`HF_HOME` diarahkan ke Drive** — bobot model diunduh sekali, tidak hilang saat restart.
> - **DINOv3 diekstrak fp32** (via `embed.py`) — fp16 bikin NaN, itu sebab DINOv3 hilang dari cache.
>
> **PRASYARAT — DINOv3 gated.** Terima lisensi di `facebook/dinov3-vitl16-pretrain-lvd1689m`
> dan `facebook/dinov3-convnext-base-pretrain-lvd1689m`, lalu simpan token sebagai
> Colab Secret `HF_TOKEN` (ikon 🔑, aktifkan "Notebook access"). Jangan paste token di cell.

---
## 🔧 SETUP

In [ ]:
# Cell 1 — HF_HOME ke Drive (WAJIB paling awal, sebelum import transformers/hf apa pun)
# Kalau di-set setelah library HF ter-import, TIDAK ngefek: cache tetap ke /root
# (hilang tiap restart -> download berulang -> kena rate limit). Makanya sel ini
# duluan, sebelum mount pun tidak apa (env var murni, tak butuh Drive terpasang).
import os
os.environ["HF_HOME"] = "/content/drive/MyDrive/BDC2026apace/hf_cache"
print("HF_HOME =", os.environ["HF_HOME"])

In [ ]:
# Cell 2 — Clone / update repo
import os
if not os.path.exists('/content/satria-data-bdcugm02'):
    !git clone https://github.com/agaggigit/satria-data-bdcugm02.git
else:
    !git -C /content/satria-data-bdcugm02 pull
print('✅ Repo siap')

In [ ]:
# Cell 3 — Dependencies. hf_xet DIPASANG, bukan di-uninstall.
# huggingface_hub 1.x memakai backend xet sebagai DEFAULT; HF_HUB_DISABLE_XET
# tidak lagi efektif. Tanpa paket hf_xet, download jatuh ke fallback rusak
# -> "SignatureError: invalid key pair id". Jadi hf_xet WAJIB ADA.
!pip install -q transformers scikit-learn lightgbm hf_xet

import importlib.util
assert importlib.util.find_spec("hf_xet"), "❌ hf_xet TIDAK terpasang — download akan gagal"
print('✅ dependencies siap (hf_xet TERPASANG)')

In [ ]:
# Cell 4 — HF token (DINOv3 gated). Dari Colab Secrets, BUKAN di-hardcode.
import os
from google.colab import userdata

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
print('✅ HF_TOKEN ter-set' if os.environ.get('HF_TOKEN') else '❌ HF_TOKEN KOSONG')

In [ ]:
# Cell 5 — Mount Drive + verifikasi GPU + konfirmasi HF cache benar-benar ke Drive
import os, torch
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '❌ TIDAK ADA GPU')

# Bukti HF_HOME kepakai (bukan cuma ter-set): cache harus menunjuk ke Drive.
from huggingface_hub.constants import HF_HUB_CACHE
print('HF cache dipakai:', HF_HUB_CACHE)
assert HF_HUB_CACHE.startswith('/content/drive'), \
    '❌ HF cache TIDAK di Drive — HF_HOME di-set setelah import HF. Restart runtime, jalankan Cell 1 PALING awal.'

for p in [
    '/content/drive/MyDrive/BDC2026apace/output_trackA/folds.csv',
    '/content/drive/MyDrive/BDC2026apace/submission.csv',
    '/content/drive/MyDrive/BDC2026apace/test',
    '/content/drive/MyDrive/BDC2026apace/output_trackB/embeddings',
]:
    print(('✅' if os.path.exists(p) else '❌ TIDAK ADA'), p)

---
## 🧹 BERSIH-BERSIH (sekali, sebelum ekstraksi)

In [ ]:
# Cell 6 — Bersihkan duplikat Drive "(1).npy" + shard DINOv3 basi.
# Drive membuat "nama (1).npy" saat file ditulis dua kali -> probe_grid bisa baca
# yang basi tanpa sadar. Kita HAPUS varian "(1)" HANYA kalau file aslinya ada
# (artinya "(1)" memang duplikat, bukan satu-satunya salinan).
import os, glob, re

EMB = '/content/drive/MyDrive/BDC2026apace/output_trackB/embeddings'

dups = glob.glob(os.path.join(EMB, '* (*).npy')) + glob.glob(os.path.join(EMB, '* (*).json'))
removed = 0
for d in dups:
    base = re.sub(r' \(\d+\)(\.\w+)$', r'\1', d)   # "x (1).npy" -> "x.npy"
    if os.path.exists(base):
        os.remove(d); removed += 1
        print('hapus duplikat:', os.path.basename(d))
    else:
        print('⚠️  SIMPAN (tak ada aslinya):', os.path.basename(d))
print(f'{removed} duplikat dihapus')

# Shard DINOv3 setengah jadi dari run fp16 yang NaN -> buang biar merge tak baca sampah.
stale = glob.glob(os.path.join(EMB, 'dinov3*.part*'))
for s in stale:
    os.remove(s); print('hapus shard basi:', os.path.basename(s))
print(f'{len(stale)} shard basi dihapus')

---
## 🚀 EKSTRAKSI

1. **Preflight** — load kelima backbone, cek masing-masing keluar embedding. DINOv3 fp32.
2. **Copy gambar ke disk lokal** — paralel, sekali, resume-safe.
3. **Ekstraksi** per backbone. SigLIP yang sudah ada di-skip; DINOv3 diekstrak fp32 (tak NaN).

In [ ]:
# Cell 7 — Jalankan extract_all.py (GPU). SigLIP di-skip, DINOv3 fp32.
# Kalau putus di tengah, jalankan ULANG sel ini — resume otomatis (shard).
import os
os.chdir('/content/satria-data-bdcugm02/track_b/src')
!python ../experiments/extract_all.py

In [ ]:
# Cell 8 — Verifikasi 20 file ADA di Drive + tak ada NaN.
import os, sys, numpy as np
sys.path.insert(0, '/content/satria-data-bdcugm02/track_b/src')
from config import CFG

NAMES = ['siglip2b256', 'siglip2so400m', 'siglip1b256', 'dinov3vitl', 'dinov3cnxb']
missing, bad = [], []
for name in NAMES:
    for suffix in ['', '_tta']:
        for split in ['train', 'test']:
            fname = f'{name}{suffix}_{split}.npy'
            path = os.path.join(CFG.embeddings_dir, fname)
            if not os.path.exists(path):
                missing.append(fname); print('HILANG', fname); continue
            e = np.load(path)
            nan = (not np.isfinite(e).all())
            if nan: bad.append(fname)
            print('OK  ' if not nan else '⚠️ NaN', f'{fname:32s} shape={e.shape}')

print()
if missing: print(len(missing), 'HILANG — jalankan ulang Cell 7 (resume otomatis)')
if bad:     print(len(bad), 'mengandung NaN — cek resolve_dtype di embed.py')
if not missing and not bad: print('✅ LENGKAP 20 file, tak ada NaN — lanjut ke probe_grid.py')